# 01 Data Preparation

Цель ноутбука — загрузить исходный Online Retail II dataset, привести его к удобному формату, очистить транзакции и подготовить признаки для когортного анализа.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    add_cohort_features,
    add_synthetic_acquisition_channel,
    clean_transactions,
    load_online_retail_data,
    normalize_columns,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print(f"Project root: {PROJECT_ROOT}")

## 2. Load raw data

Файл должен лежать в `data/raw/`. Загрузчик поддерживает CSV, XLSX, XLS и ZIP, поэтому точное имя файла не требуется.

In [ ]:
raw_df = load_online_retail_data(RAW_PATH)

print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())

## 3. Normalize columns

In [ ]:
normalized_df = normalize_columns(raw_df)

print("Normalized columns:")
print(list(normalized_df.columns))
display(normalized_df.head())

## 4. Clean transactions

Удаляем строки без клиента или даты, отмененные инвойсы, неположительные количества и цены, а также дубликаты. После очистки рассчитываем `revenue`.

In [ ]:
rows_before_cleaning = len(normalized_df)
cleaned_df = clean_transactions(normalized_df)
rows_after_cleaning = len(cleaned_df)
removed_rows = rows_before_cleaning - rows_after_cleaning
removed_share = removed_rows / rows_before_cleaning if rows_before_cleaning else 0

print(f"Shape before cleaning: {normalized_df.shape}")
print(f"Shape after cleaning: {cleaned_df.shape}")
print(f"Removed rows: {removed_rows:,}")
print(f"Removed share: {removed_share:.2%}")
display(cleaned_df.head())

## 5. Add synthetic acquisition channel

`acquisition_channel` отсутствует в исходном датасете, поэтому ниже создается искусственная переменная на уровне клиента. Она нужна только для демонстрации сегментации.

In [ ]:
transactions_with_channel = add_synthetic_acquisition_channel(cleaned_df, random_state=42)

channel_check = transactions_with_channel.groupby("customer_id")["acquisition_channel"].nunique()
print(f"Customers with more than one channel: {(channel_check > 1).sum()}")

display(
    transactions_with_channel[["customer_id", "acquisition_channel"]]
    .drop_duplicates()
    ["acquisition_channel"]
    .value_counts(normalize=True)
    .rename("customers_share")
    .to_frame()
)
display(transactions_with_channel.head())

## 6. Create cohort features

In [ ]:
prepared_df = add_cohort_features(transactions_with_channel)

cohort_columns = [
    "customer_id",
    "invoice",
    "invoice_date",
    "order_date",
    "order_month",
    "first_purchase_date",
    "cohort_month",
    "days_since_first_purchase",
    "months_since_first_purchase",
    "acquisition_channel",
    "revenue",
]

display(prepared_df[cohort_columns].head())

## 7. Save processed dataset

In [ ]:
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
prepared_file = PROCESSED_PATH / "transactions_prepared.csv"
prepared_df.to_csv(prepared_file, index=False)

print(f"Saved prepared transactions to: {prepared_file}")

## 8. Data quality summary

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "raw_rows",
            "prepared_rows",
            "removed_rows",
            "removed_share",
            "date_min",
            "date_max",
            "customers",
            "orders",
            "countries",
            "total_revenue",
        ],
        "value": [
            rows_before_cleaning,
            rows_after_cleaning,
            removed_rows,
            f"{removed_share:.2%}",
            prepared_df["invoice_date"].min(),
            prepared_df["invoice_date"].max(),
            prepared_df["customer_id"].nunique(),
            prepared_df["invoice"].nunique(),
            prepared_df["country"].nunique(),
            prepared_df["revenue"].sum(),
        ],
    }
)

display(summary)

## 9. Key findings

- Подготовлен воспроизводимый датасет для дальнейшего когортного анализа.
- Отмененные заказы, строки без клиента и невалидные транзакции удалены.
- `acquisition_channel` добавлен синтетически и должен интерпретироваться только как демонстрационный сегмент.
- Следующий шаг — расчет monthly retention и D7/D30/D90 retention по когортам.